# Notebook 5: Backtesting — Kupiec POF & Basel Traffic Light
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '../src')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})

returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 observations | 2015-01-01 to 2024-12-31


In [2]:
from var_models import historical_var
from backtesting import compute_exceptions, kupiec_pof_test, basel_traffic_light, full_backtest
var_rolling = historical_var(port, window=252)
var_bt = var_rolling['VaR_99pct']
actual_bt = port.loc[var_bt.index]
exceptions = compute_exceptions(actual_bt, var_bt)
n_obs=len(exceptions); n_exc=int(exceptions.sum())
print(f'Observations  : {n_obs:,}')
print(f'Exceptions    : {n_exc}')
print(f'Exception rate: {n_exc/n_obs*100:.2f}% (expected 1.00%)')

Observations  : 2,358
Exceptions    : 22
Exception rate: 0.93% (expected 1.00%)


In [3]:
kupiec = kupiec_pof_test(n_obs, n_exc, 0.99)
print('Kupiec POF Test:')
for k,v in kupiec.items(): print(f'  {k}: {v}')

Kupiec POF Test:
  test: Kupiec POF
  n_obs: 2358
  n_exceptions: 22
  exception_rate: 0.0093
  expected_rate: 0.01
  LR_statistic: 0.1094
  critical_value: 3.8415
  p_value: 0.7408
  reject_H0: False
  model_valid: True


In [4]:
basel = basel_traffic_light(n_exc)
print('Basel Traffic Light:')
for k,v in basel.items(): print(f'  {k}: {v}')

Basel Traffic Light:
  zone: RED
  n_exceptions: 22
  capital_addon: 4
  action: Model rejected — revise immediately


In [5]:
fig,axes=plt.subplots(2,1,figsize=(13,9))
ax1=axes[0]
ax1.plot(actual_bt.index,actual_bt,color='#888',linewidth=0.6,alpha=0.7,label='Portfolio return')
ax1.plot(var_bt.index,-var_bt,color='#185FA5',linewidth=1.3,label='99% VaR')
exc_dates=exceptions[exceptions==1].index
ax1.scatter(exc_dates,actual_bt[exc_dates],color='#E24B4A',zorder=5,s=25,label=f'Exceptions ({n_exc})')
ax1.set_title('VaR Backtesting — 99% Historical VaR Exceptions'); ax1.legend(fontsize=9)
ax2=axes[1]
roll_exc=exceptions.rolling(252).sum()
ax2.plot(roll_exc.index,roll_exc,color='#7B1FA2',linewidth=1.3)
ax2.axhline(4,color='#1D9E75',linewidth=1.3,linestyle='--',label='Basel Green (<=4)')
ax2.axhline(9,color='#F57C00',linewidth=1.3,linestyle='--',label='Basel Yellow (<=9)')
ax2.axhline(10,color='#E24B4A',linewidth=1.3,linestyle='--',label='Basel Red (>=10)')
ax2.set_title('Rolling 252-Day Exception Count'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/06_backtesting_exceptions.png',dpi=150,bbox_inches='tight')
plt.show()
exc_report=pd.DataFrame({'Portfolio Return':actual_bt,'VaR 99%':-var_bt,'Exception':exceptions})
exc_report.to_csv('../results/09_var_exception_report.csv')
print('Exception report saved.')

Exception report saved.
